# 02 - Customer, Seller, and Geolocation EDA and Preprocessing

**Member B contribution:** customers, sellers, geolocation, geographical exploratory data analysis, preprocessing, and order-level geographic feature engineering.

This notebook is the working record for Member B's analysis. It will document assumptions, validation evidence, preprocessing decisions, and the production of geography-related datasets for downstream integration.

## 2. Business Problem and Prediction Objective

The project aims to predict whether an Olist order will be delivered late. Member B contributes customer, seller, and geographic information that may explain delivery risk, such as distance proxies, state and region relationships, and the geographic distribution of sellers linked to an order.

Member A owns target engineering. This notebook will consume the agreed target only after Member A makes it available; it will not define or modify the target.

## 3. Scope and Ownership Boundaries

**In scope for Member B**

- Audit and preprocess customer, seller, and geolocation data.
- Create stable geographic attributes and Brazilian region features.
- Enrich customers and sellers with cleaned location data.
- Create order-level geographic features for the final integration step.

**Ownership boundaries**

- Raw files are read-only and must never be overwritten.
- Orders and order_items are used only as relationship bridges.
- Member A owns target engineering.
- Member C owns order-item and product preprocessing.
- Member D owns the final overall merge.
- This notebook will not modify another member's notebook, processed output, or shared source code.

## 4. Library Imports and Path Configuration

Imports and repository-relative path configuration will be added here. Paths will be resolved from the repository structure so that the notebook does not depend on a machine-specific absolute path. Randomness, display settings, and plotting conventions will be declared explicitly where relevant.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def find_repository_root(start: Path) -> Path:
    """Find the repository root when launched from the root or notebooks directory."""
    resolved_start = start.resolve()
    for candidate in (resolved_start, *resolved_start.parents):
        if (candidate / "data" / "raw").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a repository containing data/raw and notebooks.")


REPO_ROOT = find_repository_root(Path.cwd())
RAW_DIR = REPO_ROOT / "data" / "raw"

display(pd.DataFrame({"resolved_path": [str(REPO_ROOT), str(RAW_DIR)]}, index=["repository_root", "raw_data_directory"]))

,resolved_path
repository_root,C:\Users\Hasini\OneDrive\Desktop\FDM project\O...
raw_data_directory,C:\Users\Hasini\OneDrive\Desktop\FDM project\O...


**Interpretation.** The repository root was resolved successfully from the active working directory, and all raw inputs are referenced through the repository-relative `data/raw` path. No machine-specific project path is embedded in the notebook.

## 5. Dataset Loading

The customer, seller, and geolocation tables will be loaded from `data/raw/` without modification. The orders and order_items tables may be read later only to connect customer and seller records to an order. Loading checks will confirm file presence, parse success, row counts, and expected columns before analysis begins.

In [2]:
CUSTOMERS_PATH = RAW_DIR / "olist_customers_dataset.csv"
SELLERS_PATH = RAW_DIR / "olist_sellers_dataset.csv"
GEOLOCATION_PATH = RAW_DIR / "olist_geolocation_dataset.csv"

required_paths = [CUSTOMERS_PATH, SELLERS_PATH, GEOLOCATION_PATH]
missing_files = [path.name for path in required_paths if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing required raw files: {missing_files}")

customers = pd.read_csv(
    CUSTOMERS_PATH,
    dtype={"customer_zip_code_prefix": "string"},
)
sellers = pd.read_csv(
    SELLERS_PATH,
    dtype={"seller_zip_code_prefix": "string"},
)
geolocation = pd.read_csv(
    GEOLOCATION_PATH,
    dtype={"geolocation_zip_code_prefix": "string"},
)

datasets = {
    "customers": customers,
    "sellers": sellers,
    "geolocation": geolocation,
}
zip_columns = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix",
}

load_summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(frame),
            "columns": frame.shape[1],
            "zip_dtype": str(frame[zip_columns[name]].dtype),
            "five_character_zip_rows": int(frame[zip_columns[name]].str.len().eq(5).sum()),
        }
        for name, frame in datasets.items()
    ]
).set_index("table")
display(load_summary)

,rows,columns,zip_dtype,five_character_zip_rows
table,,,,
customers,99441,5,string,99441
sellers,3095,4,string,3095
geolocation,1000163,5,string,1000163


**Interpretation.** The three authorized raw tables loaded successfully: customers has **99,441 rows and 5 columns**, sellers has **3,095 rows and 4 columns**, and geolocation has **1,000,163 rows and 5 columns**. Every ZIP-prefix value was loaded with pandas' string dtype and all **1,102,699** rows across the tables retain a five-character prefix, including leading zeros. Orders and order_items were not loaded.

## 6. Data Dictionary and Table Grain

This section will define each source table's business meaning, expected grain, identifiers, geographic fields, and planned use. Particular attention will be given to the distinction between `customer_id` and `customer_unique_id`, the seller identifier, geolocation ZIP-code prefixes, and one-to-many relationships introduced by order items.

In [3]:
data_dictionary = pd.DataFrame(
    [
        {
            "table": "customers",
            "expected_grain": "one row per order-specific customer_id",
            "key_fields": "customer_id; customer_unique_id",
            "geographic_fields": "customer_zip_code_prefix; customer_city; customer_state",
        },
        {
            "table": "sellers",
            "expected_grain": "one row per seller_id",
            "key_fields": "seller_id",
            "geographic_fields": "seller_zip_code_prefix; seller_city; seller_state",
        },
        {
            "table": "geolocation",
            "expected_grain": "one row per observed ZIP-prefix coordinate record",
            "key_fields": "geolocation_zip_code_prefix (non-unique)",
            "geographic_fields": "geolocation_lat; geolocation_lng; geolocation_city; geolocation_state",
        },
    ]
).set_index("table")
display(data_dictionary)

,expected_grain,key_fields,geographic_fields
table,,,
customers,one row per order-specific customer_id,customer_id; customer_unique_id,customer_zip_code_prefix; customer_city; custo...
sellers,one row per seller_id,seller_id,seller_zip_code_prefix; seller_city; seller_state
geolocation,one row per observed ZIP-prefix coordinate record,geolocation_zip_code_prefix (non-unique),geolocation_lat; geolocation_lng; geolocation_...


**Interpretation.** The tables have different grains. `customer_id` identifies an order-specific customer record, while `customer_unique_id` links repeat purchases by the same customer. Sellers are expected to be unique by `seller_id`. Geolocation is an observation table, so its ZIP prefix is expected to repeat and cannot be treated as a primary key without later aggregation.

## 7. Initial Structural Audit

The initial audit will record shapes, column names, data types, representative records, identifier coverage, and basic descriptive statistics. Schema differences from the documented Olist data model will be flagged before transformations are considered.

In [4]:
structural_summary = pd.DataFrame(
    [
        {
            "table": name,
            "shape": str(frame.shape),
            "columns": ", ".join(frame.columns),
        }
        for name, frame in datasets.items()
    ]
).set_index("table")
display(Markdown("### Shapes and columns"), structural_summary)

dtype_summary = pd.concat(
    {name: frame.dtypes.astype(str).rename("dtype") for name, frame in datasets.items()},
    names=["table", "column"],
)
display(Markdown("### Data types"), dtype_summary.to_frame())

for name, frame in datasets.items():
    display(Markdown(f"### First five rows: {name}"), frame.head(5))

### Shapes and columns

,shape,columns
table,,
customers,"(99441, 5)","customer_id, customer_unique_id, customer_zip_..."
sellers,"(3095, 4)","seller_id, seller_zip_code_prefix, seller_city..."
geolocation,"(1000163, 5)","geolocation_zip_code_prefix, geolocation_lat, ..."


### Data types

dtype
table       column                              
customers   customer_id                      str
            customer_unique_id               str
            customer_zip_code_prefix      string
            customer_city                    str
            customer_state                   str
sellers     seller_id                        str
            seller_zip_code_prefix        string
            seller_city                      str
            seller_state                     str
geolocation geolocation_zip_code_prefix   string
            geolocation_lat              float64
            geolocation_lng              float64
            geolocation_city                 str
            geolocation_state                str

### First five rows: customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


### First five rows: sellers

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


### First five rows: geolocation

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,01037,-23.5456,-46.6393,sao paulo,SP
1,01046,-23.5461,-46.6448,sao paulo,SP
2,01046,-23.5461,-46.6430,sao paulo,SP
3,01041,-23.5444,-46.6395,sao paulo,SP
4,01035,-23.5416,-46.6416,sao paulo,SP


**Interpretation.** All expected columns are present. Identifier, city, and state fields loaded as strings; all three ZIP-prefix fields loaded as dedicated string columns; and geolocation latitude and longitude loaded as floating-point values. The first five records confirm the expected table layouts without requiring any mutation of the source DataFrames.

In [5]:
for name, frame in datasets.items():
    display(
        Markdown(f"### Descriptive statistics: {name}"),
        frame.describe(include="all").transpose(),
    )

### Descriptive statistics: customers

,count,unique,top,freq
customer_id,99441,99441,06b8999e2fba1a1fbc88172c00ba8bc7,1
customer_unique_id,99441,96096,8d50f5eadf50201ccdcedfb9e2ac8455,17
customer_zip_code_prefix,99441,14994,22790,142
customer_city,99441,4119,sao paulo,15540
customer_state,99441,27,SP,41746


### Descriptive statistics: sellers

,count,unique,top,freq
seller_id,3095,3095,3442f8959a84dea7ee197c632cb2df15,1
seller_zip_code_prefix,3095,2246,14940,49
seller_city,3095,611,sao paulo,694
seller_state,3095,23,SP,1849


### Descriptive statistics: geolocation

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
geolocation_zip_code_prefix,1000163,19015,24220,1146,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geolocation_lat,"1,000,163.0000",NaN,NaN,NaN,-21.1762,5.7159,-36.6054,-23.6035,-22.9194,-19.9796,45.0659
geolocation_lng,"1,000,163.0000",NaN,NaN,NaN,-46.3905,4.2697,-101.4668,-48.5732,-46.6379,-43.7677,121.1054
geolocation_city,1000163,8011,sao paulo,135800,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geolocation_state,1000163,27,SP,404268,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Interpretation.** The descriptive summaries cover every column, including categorical frequency statistics and the coordinate distribution. They confirm full non-null counts at this stage and show that high-frequency cities and states dominate each source, while the geolocation coordinates vary across a broad spatial range. These are audit observations only; no outlier rule or cleaning decision is applied here.

## 8. Missing-Value Analysis

Missingness will be measured by count and percentage for every relevant field. The analysis will distinguish true missing data from failed geographic matches and will assess whether missingness is systematic by state, city, ZIP-code prefix, or table source. No imputation strategy will be applied before its effect and modelling boundary are documented.

In [6]:
missing_summary = pd.concat(
    {
        name: pd.DataFrame(
            {
                "missing_count": frame.isna().sum(),
                "missing_percentage": frame.isna().mean().mul(100),
            }
        )
        for name, frame in datasets.items()
    },
    names=["table", "column"],
)
display(missing_summary)

missing_count  missing_percentage
table       column                                                        
customers   customer_id                              0              0.0000
            customer_unique_id                       0              0.0000
            customer_zip_code_prefix                 0              0.0000
            customer_city                            0              0.0000
            customer_state                           0              0.0000
sellers     seller_id                                0              0.0000
            seller_zip_code_prefix                   0              0.0000
            seller_city                              0              0.0000
            seller_state                             0              0.0000
geolocation geolocation_zip_code_prefix              0              0.0000
            geolocation_lat                          0              0.0000
            geolocation_lng                          0              0.0000
            geolocation_city                         0              0.0000
            geolocation_state                        0              0.0000

**Interpretation.** All audited columns contain **0 missing values (0.00%)** in customers, sellers, and geolocation. This does not guarantee that every ZIP prefix will match during later enrichment; join coverage will be assessed separately. Nothing is imputed in this audit.

## 9. Duplicate and Key-Uniqueness Analysis

This section will test expected primary-key uniqueness, exact duplicate rows, conflicting records, and multiplicity at each join key. Geolocation duplicates will be examined as repeated observations rather than removed automatically, because multiple coordinates can legitimately share a ZIP-code prefix.

In [7]:
duplicate_summary = pd.DataFrame(
    {
        "row_count": {name: len(frame) for name, frame in datasets.items()},
        "exact_duplicate_count": {name: int(frame.duplicated().sum()) for name, frame in datasets.items()},
        "exact_duplicate_percentage": {
            name: frame.duplicated().mean() * 100 for name, frame in datasets.items()
        },
    }
)
display(duplicate_summary)

,row_count,exact_duplicate_count,exact_duplicate_percentage
customers,99441,0,0.0000
sellers,3095,0,0.0000
geolocation,1000163,261831,26.1788


**Interpretation.** Customers and sellers each contain **0 exact duplicate rows**. Geolocation contains **261,831 exact duplicate rows**, equal to **26.18%** of its 1,000,163 rows. These records are only identified here; none are dropped during the audit.

In [8]:
unique_value_counts = pd.concat(
    {name: frame.nunique(dropna=False).rename("unique_values") for name, frame in datasets.items()},
    names=["table", "column"],
).to_frame()
display(Markdown("### Unique values by column"), unique_value_counts)

location_cardinality = pd.DataFrame(
    [
        {
            "table": "customers",
            "state_cardinality": customers["customer_state"].nunique(),
            "city_cardinality": customers["customer_city"].nunique(),
        },
        {
            "table": "sellers",
            "state_cardinality": sellers["seller_state"].nunique(),
            "city_cardinality": sellers["seller_city"].nunique(),
        },
        {
            "table": "geolocation",
            "state_cardinality": geolocation["geolocation_state"].nunique(),
            "city_cardinality": geolocation["geolocation_city"].nunique(),
        },
    ]
).set_index("table")
display(Markdown("### State and city cardinality"), location_cardinality)

### Unique values by column

unique_values
table       column                                    
customers   customer_id                          99441
            customer_unique_id                   96096
            customer_zip_code_prefix             14994
            customer_city                         4119
            customer_state                          27
sellers     seller_id                             3095
            seller_zip_code_prefix                2246
            seller_city                            611
            seller_state                            23
geolocation geolocation_zip_code_prefix          19015
            geolocation_lat                     717360
            geolocation_lng                     717613
            geolocation_city                      8011
            geolocation_state                       27

### State and city cardinality

,state_cardinality,city_cardinality
table,,
customers,27,4119
sellers,23,611
geolocation,27,8011


**Interpretation.** Customers span **27 states and 4,119 cities**; sellers span **23 states and 611 cities**; and geolocation spans **27 states and 8,011 city labels**. City cardinality is substantially higher than state cardinality, especially in geolocation, and will require quality review before any later encoding decision.

In [9]:
customer_unique_frequency = customers["customer_unique_id"].value_counts(dropna=False)
geolocation_zip_frequency = geolocation["geolocation_zip_code_prefix"].value_counts(dropna=False)

key_audit = pd.DataFrame(
    [
        {
            "check": "customer_id",
            "rows": len(customers),
            "unique_values": customers["customer_id"].nunique(dropna=False),
            "is_unique": customers["customer_id"].is_unique,
            "repeated_keys": int(customers["customer_id"].duplicated(keep=False).groupby(customers["customer_id"]).any().sum()),
            "rows_in_repeated_keys": int(customers["customer_id"].duplicated(keep=False).sum()),
            "maximum_repetitions": int(customers["customer_id"].value_counts(dropna=False).max()),
        },
        {
            "check": "customer_unique_id",
            "rows": len(customers),
            "unique_values": customers["customer_unique_id"].nunique(dropna=False),
            "is_unique": customers["customer_unique_id"].is_unique,
            "repeated_keys": int(customer_unique_frequency.gt(1).sum()),
            "rows_in_repeated_keys": int(customer_unique_frequency[customer_unique_frequency.gt(1)].sum()),
            "maximum_repetitions": int(customer_unique_frequency.max()),
        },
        {
            "check": "seller_id",
            "rows": len(sellers),
            "unique_values": sellers["seller_id"].nunique(dropna=False),
            "is_unique": sellers["seller_id"].is_unique,
            "repeated_keys": int(sellers["seller_id"].duplicated(keep=False).groupby(sellers["seller_id"]).any().sum()),
            "rows_in_repeated_keys": int(sellers["seller_id"].duplicated(keep=False).sum()),
            "maximum_repetitions": int(sellers["seller_id"].value_counts(dropna=False).max()),
        },
        {
            "check": "geolocation_zip_code_prefix",
            "rows": len(geolocation),
            "unique_values": geolocation["geolocation_zip_code_prefix"].nunique(dropna=False),
            "is_unique": geolocation["geolocation_zip_code_prefix"].is_unique,
            "repeated_keys": int(geolocation_zip_frequency.gt(1).sum()),
            "rows_in_repeated_keys": int(geolocation_zip_frequency[geolocation_zip_frequency.gt(1)].sum()),
            "maximum_repetitions": int(geolocation_zip_frequency.max()),
        },
    ]
).set_index("check")
display(key_audit)

,rows,unique_values,is_unique,repeated_keys,rows_in_repeated_keys,maximum_repetitions
check,,,,,,
customer_id,99441,99441,True,0,0,1
customer_unique_id,99441,96096,False,2997,6342,17
seller_id,3095,3095,True,0,0,1
geolocation_zip_code_prefix,1000163,19015,False,17972,999120,1146


**Interpretation.** All **99,441 customer_id values** and all **3,095 seller_id values** are unique. The customer table contains **96,096 customer_unique_id values**; **2,997** of them repeat, covering **6,342 rows**, and the highest repetition count is **17**. Geolocation contains **19,015 ZIP prefixes**; **17,972** repeat, covering **999,120 rows**, with a maximum of **1,146 observations** for one prefix. The geolocation ZIP prefix therefore requires aggregation before it can support a many-to-one enrichment join.

## 10. Customer EDA

Customer EDA will examine record grain, unique-customer behavior, state and city distributions, ZIP-code coverage, geographic concentration, and unusual or inconsistent location values. Results will be presented with counts and normalized proportions where appropriate.

## 11. Seller EDA

Seller EDA will assess seller uniqueness, state and city distributions, ZIP-code coverage, geographic concentration, and data-quality anomalies. Any order-linked seller activity measure will be treated as contextual analysis and will respect the order_items ownership boundary.

## 12. Geolocation EDA

Geolocation EDA will examine coordinate ranges, ZIP-code prefix multiplicity, city and state consistency, spatial coverage, repeated coordinates, and implausible latitude or longitude values. Visual analysis will focus on interpretable geographic patterns and evidence needed to choose a ZIP-level aggregation rule.

## 13. Data-Quality Observations

Confirmed quality issues, their scope, likely causes, and chosen responses will be summarized here. Observations will remain separate from transformations so that every preprocessing decision has traceable evidence.

## 14. Geolocation Preprocessing

A reproducible ZIP-code-prefix-level geolocation table will be designed here. Candidate aggregation and consistency rules will be justified using the earlier audit, with explicit handling for invalid coordinates, duplicate observations, conflicting city or state labels, and unresolved prefixes.

## 15. Customer and Seller Location Enrichment

Customers and sellers will be joined independently to the cleaned ZIP-level geolocation table. Join cardinality, row preservation, match rates, unmatched prefixes, suffix conventions, and post-merge uniqueness will be validated before either enriched table is saved.

## 16. Brazilian Region Feature Engineering

Validated state codes will be mapped to Brazil's five official regions: North, Northeast, Central-West, Southeast, and South. The mapping will be defined once and tested for full coverage, with unknown or invalid states retained as an explicit unresolved category rather than silently assigned.

## 17. Order-Level Geographic Feature Engineering

The orders table will bridge orders to customers, and order_items will bridge orders to sellers. These source files remain read-only and will not be preprocessed here. Planned order-level features may include customer-seller state or region agreement, seller counts, cross-state or cross-region indicators, coordinate availability, and justified distance summaries.

Aggregation must preserve one row per `order_id` and avoid duplicating order-item or product transformations owned by Member C. The resulting feature table will be handed to Member D for the final overall merge.

## 18. Target-Related EDA - Pending Member A's Target

Target-based comparisons are intentionally deferred until Member A supplies the approved target and its definition. Once available, this section may compare geographic feature distributions by target class without recreating, altering, or inferring the target independently.

## 19. Leakage Analysis

Every proposed feature will be reviewed against the model's prediction timestamp. Post-outcome information, including actual delivery outcomes or transformations derived from the target, must not enter Member B outputs.

Encoding, imputation, and scaling for modelling must be fitted after the chronological split and only on the training partition. This notebook may preserve missingness and raw categorical values for downstream fitting rather than learn full-dataset modelling parameters.

## 20. Output Validation

Each processed output will be checked for expected grain, key uniqueness, row counts, schema, missingness, coordinate validity, join coverage, duplicate columns, and reproducibility. Order-level output must contain exactly one row per order represented within Member B's agreed bridge scope.

## 21. Findings and Preprocessing Decisions

Key findings and final decisions will be summarized with supporting evidence and downstream implications. Material conclusions will also be recorded in `report/member_b_findings_and_decisions.md` when implementation is authorized.

## 22. Saving Processed Outputs

After validation, this notebook is expected to create only the following Member B outputs:

- `data/processed/geolocation_zip_clean.csv`
- `data/processed/customers_geo_clean.csv`
- `data/processed/sellers_geo_clean.csv`
- `data/processed/order_geo_features.csv`

Saving will be explicit and will never overwrite raw CSV files. No output is produced by the current notebook scaffold.

## 23. Member B Viva Summary

This section will provide a concise explanation of the source-table grains, main geographic quality issues, preprocessing rationale, engineered features, ownership boundaries, leakage controls, validation results, and limitations. It will be completed after the analysis and preprocessing have been implemented.